## Milestone 3: AI-Powered Test Case Generator for Software Testing
Class: "DSC680-T301 Advance Uses of Generative AI (2265-1)<br>
Student: "Roshan GC" <br>
Professor: "Frank Neugebauer" <br>
Date: "05/10/2026"

## Purpose
This notebook follows from previous Milestone-2 and it aims to build a fine-tuned OpenAI model for producing software test cases. An objective is to move from past prompt experiments by training a model on QA examples such that the results are more consistent with the practices of software testing.
This notebook covers data set creation, uploading files, creating fine-tune jobs, endpoints tracking, metrics analysis, testing model, and commentary.

## Import required libraries and setting up the OpenAI client

In [2]:
# importing required libraries
# Used Car Price Prediction - Milestone 2 / Project Draft

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Load dataset
df = pd.read_csv("/users/roshan/DSC680/CAR_DETAILS_FROM_CAR_DEKHO.csv")

# Preview data
print(df.head())
print(df.info())
print(df.describe())
print(df.isnull().sum())

ModuleNotFoundError: No module named 'pandas'

## Dataset
For the fine-tuning, I generated a small QA-oriented dataset. Each entry has a software requirement as input and structured test cases as output. The dataset trains the model on how to generate test case IDs, scenario descriptions, categories, steps etc.

In [28]:
# Defining the system message
system_message = (
    "You are a QA test case generation assistant. "
    "Given a software requirement, generate concise, structured test cases. "
    "Use markdown table format with columns: TC ID, Scenario, Category, Steps, Expected Result."
)

# Sample dataset
examples = [
    {
        "requirement": "Login feature: username and password are required. Valid credentials allow login. Invalid credentials show an error. Account locks after five failed attempts.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_LOGIN_001 | Login with valid credentials | Functional | Open login page; enter valid username; enter valid password; click Login | User logs in successfully and is redirected to dashboard |\n| TC_LOGIN_002 | Submit blank username and password | Negative | Open login page; leave username blank; leave password blank; click Login | Required field validation messages are displayed |\n| TC_LOGIN_003 | Login with invalid password | Negative | Enter valid username; enter invalid password; click Login | Invalid credentials error is displayed |\n| TC_LOGIN_004 | Lock account after five failed attempts | Boundary | Attempt login five times with wrong password | Account is locked after the fifth failed attempt |\n| TC_LOGIN_005 | Login after account is locked | Negative | Use locked account credentials; click Login | Login is denied and locked account message is displayed |"
    },
    {
        "requirement": "Registration form: email, password, and confirm password are required. Email must be valid. Password must be at least eight characters. Password and confirm password must match.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_REG_001 | Register with valid details | Functional | Open registration page; enter valid email; enter valid password; confirm password; submit | Account is created successfully |\n| TC_REG_002 | Submit empty registration form | Negative | Open registration page; leave all fields blank; submit | Required field messages are displayed |\n| TC_REG_003 | Register with invalid email format | Negative | Enter invalid email; enter valid matching passwords; submit | Email format validation message is displayed |\n| TC_REG_004 | Password below minimum length | Boundary | Enter valid email; enter password with seven characters; confirm same password; submit | Password length validation message is displayed |\n| TC_REG_005 | Password and confirm password mismatch | Negative | Enter valid email; enter different password values; submit | Password mismatch error is displayed |"
    },
    {
        "requirement": "Shopping cart: user can add products, update quantity, remove products, and view total price.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_CART_001 | Add product to cart | Functional | Open product page; click Add to Cart | Product appears in cart |\n| TC_CART_002 | Increase product quantity | Functional | Add product to cart; increase quantity | Quantity and total price update correctly |\n| TC_CART_003 | Set quantity to zero | Edge | Add product to cart; change quantity to zero | System prevents invalid quantity or removes item based on requirement |\n| TC_CART_004 | Remove product from cart | Functional | Add product to cart; click Remove | Product is removed from cart |\n| TC_CART_005 | Verify total price after multiple items | Functional | Add multiple products with different prices | Total price equals sum of item prices |"
    },
    {
        "requirement": "Password reset: user enters registered email, receives reset link, and creates a new password.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_RESET_001 | Request reset with registered email | Functional | Open reset page; enter registered email; submit | Reset link is sent |\n| TC_RESET_002 | Request reset with unregistered email | Negative | Enter unregistered email; submit | System shows appropriate message without exposing account existence |\n| TC_RESET_003 | Submit blank email | Negative | Leave email blank; submit | Required email validation message is displayed |\n| TC_RESET_004 | Use expired reset link | Edge | Open expired reset link | System rejects expired link |\n| TC_RESET_005 | Create password below minimum length | Boundary | Open reset link; enter short password; submit | Password validation message is displayed |"
    },
    {
        "requirement": "Search feature: user can search products by keyword. Results should match the search term. Empty search should show validation.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_SEARCH_001 | Search with valid keyword | Functional | Enter valid keyword; click Search | Matching products are displayed |\n| TC_SEARCH_002 | Search with no matching keyword | Negative | Enter keyword with no results; click Search | No results message is displayed |\n| TC_SEARCH_003 | Submit empty search | Negative | Leave search box blank; click Search | Validation message is displayed |\n| TC_SEARCH_004 | Search with special characters | Edge | Enter special characters; click Search | System handles input safely and does not crash |\n| TC_SEARCH_005 | Search with leading/trailing spaces | Edge | Enter keyword with spaces before and after; click Search | Search term is trimmed or handled according to requirements |"
    },
    {
        "requirement": "Checkout: user must provide shipping address, payment method, and confirm order before purchase is completed.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_CHECKOUT_001 | Complete checkout with valid data | Functional | Add item; enter shipping address; enter payment; confirm order | Order is placed successfully |\n| TC_CHECKOUT_002 | Checkout without shipping address | Negative | Add item; leave address blank; submit checkout | Address validation message is displayed |\n| TC_CHECKOUT_003 | Checkout with invalid payment details | Negative | Enter valid address; enter invalid payment; confirm order | Payment error is displayed and order is not placed |\n| TC_CHECKOUT_004 | Confirm order after cart becomes empty | Edge | Add item; remove item before confirmation; confirm order | System prevents checkout with empty cart |\n| TC_CHECKOUT_005 | Verify order summary before confirmation | Functional | Enter valid checkout data; review order summary | Summary shows correct items, taxes, shipping, and total |"
    },
    {
        "requirement": "Profile update: user can update name, phone number, and address. Required fields cannot be blank.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_PROFILE_001 | Update profile with valid details | Functional | Open profile; update name, phone, and address; save | Profile is updated successfully |\n| TC_PROFILE_002 | Save profile with blank required name | Negative | Clear name field; click Save | Required name validation message is displayed |\n| TC_PROFILE_003 | Enter invalid phone format | Negative | Enter invalid phone number; save | Phone format validation message is displayed |\n| TC_PROFILE_004 | Update address only | Functional | Change address; save | Address is updated and other fields remain unchanged |\n| TC_PROFILE_005 | Cancel profile update | Functional | Edit profile; click Cancel | Changes are discarded |"
    },
    {
        "requirement": "File upload: user can upload PDF files up to 5 MB. Other file types are not allowed.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_UPLOAD_001 | Upload valid PDF under 5 MB | Functional | Select PDF under 5 MB; upload | File uploads successfully |\n| TC_UPLOAD_002 | Upload non-PDF file | Negative | Select DOCX or JPG file; upload | File type validation message is displayed |\n| TC_UPLOAD_003 | Upload PDF larger than 5 MB | Boundary | Select PDF over 5 MB; upload | File size validation message is displayed |\n| TC_UPLOAD_004 | Upload file exactly 5 MB | Boundary | Select PDF exactly 5 MB; upload | File uploads if limit is inclusive |\n| TC_UPLOAD_005 | Upload without selecting file | Negative | Click Upload with no file selected | Required file message is displayed |"
    },
    {
        "requirement": "Order history: user can view previous orders, filter by date range, and open order details.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_ORDER_001 | View order history | Functional | Login; open Order History | Previous orders are displayed |\n| TC_ORDER_002 | Open order details | Functional | Select an order from history | Order details are displayed |\n| TC_ORDER_003 | Filter orders by valid date range | Functional | Enter valid start and end dates; apply filter | Orders within date range are displayed |\n| TC_ORDER_004 | Filter with start date after end date | Negative | Enter start date later than end date; apply filter | Date range validation message is displayed |\n| TC_ORDER_005 | No orders found for selected date range | Edge | Apply date range with no orders | No orders found message is displayed |"
    },
    {
        "requirement": "Logout feature: user can log out from the application and should not access protected pages after logout.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_LOGOUT_001 | Logout from active session | Functional | Login; click Logout | User is logged out and redirected to login page |\n| TC_LOGOUT_002 | Access protected page after logout | Negative | Logout; enter protected page URL | User is redirected to login page |\n| TC_LOGOUT_003 | Click browser back after logout | Edge | Logout; click browser Back | Protected content is not displayed |\n| TC_LOGOUT_004 | Logout when session expires | Edge | Let session expire; click Logout | System handles expired session gracefully |\n| TC_LOGOUT_005 | Verify session token invalidated | Security | Logout; reuse previous session token | Access is denied |"
    },
    {
        "requirement": "Admin user management: admin can create, edit, deactivate, and reactivate user accounts.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_ADMIN_001 | Create new user with valid data | Functional | Login as admin; open Users; enter valid user details; save | New user account is created |\n| TC_ADMIN_002 | Create user with duplicate email | Negative | Enter email already used by another user; save | Duplicate email error is displayed |\n| TC_ADMIN_003 | Edit existing user details | Functional | Open existing user; update fields; save | User details are updated |\n| TC_ADMIN_004 | Deactivate active user | Functional | Select active user; click Deactivate | User status changes to inactive |\n| TC_ADMIN_005 | Reactivate inactive user | Functional | Select inactive user; click Reactivate | User status changes to active |"
    },
    {
        "requirement": "Notification settings: user can enable or disable email and SMS notifications.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_NOTIFY_001 | Enable email notifications | Functional | Open settings; turn on email notifications; save | Email notifications are enabled |\n| TC_NOTIFY_002 | Disable email notifications | Functional | Turn off email notifications; save | Email notifications are disabled |\n| TC_NOTIFY_003 | Enable SMS without phone number | Negative | Remove phone number; enable SMS; save | Phone number required message is displayed |\n| TC_NOTIFY_004 | Enable both email and SMS | Functional | Turn on email and SMS; save | Both notification preferences are saved |\n| TC_NOTIFY_005 | Cancel changes | Functional | Change notification settings; click Cancel | Previous preferences remain unchanged |"
    },
    {
        "requirement": "Product review: logged-in users can submit a rating from 1 to 5 and optional review text.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_REVIEW_001 | Submit review with valid rating and text | Functional | Login; open product; select rating; enter review; submit | Review is submitted successfully |\n| TC_REVIEW_002 | Submit review without rating | Negative | Enter review text without selecting rating; submit | Rating required validation is displayed |\n| TC_REVIEW_003 | Submit minimum rating | Boundary | Select rating 1; submit | Review is accepted |\n| TC_REVIEW_004 | Submit maximum rating | Boundary | Select rating 5; submit | Review is accepted |\n| TC_REVIEW_005 | Submit review while logged out | Negative | Open product while logged out; attempt review | User is prompted to log in |"
    },
    {
        "requirement": "Coupon code: user can apply a valid coupon during checkout. Invalid or expired coupons should be rejected.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_COUPON_001 | Apply valid coupon | Functional | Add item to cart; enter valid coupon; apply | Discount is applied |\n| TC_COUPON_002 | Apply invalid coupon | Negative | Enter invalid coupon code; apply | Invalid coupon message is displayed |\n| TC_COUPON_003 | Apply expired coupon | Negative | Enter expired coupon code; apply | Expired coupon message is displayed |\n| TC_COUPON_004 | Apply coupon twice | Edge | Apply valid coupon; apply same coupon again | Duplicate coupon is prevented |\n| TC_COUPON_005 | Remove applied coupon | Functional | Apply coupon; click Remove | Discount is removed and total updates |"
    },
    {
        "requirement": "Appointment booking: user selects service, date, time, and confirms appointment.",
        "answer": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_APPT_001 | Book appointment with valid data | Functional | Select service; choose available date and time; confirm | Appointment is booked successfully |\n| TC_APPT_002 | Book without selecting service | Negative | Choose date and time; confirm without service | Service required validation is displayed |\n| TC_APPT_003 | Choose unavailable time slot | Negative | Select unavailable time; confirm | System prevents booking unavailable slot |\n| TC_APPT_004 | Cancel appointment before confirmation | Functional | Select appointment details; click Cancel | Appointment is not booked |\n| TC_APPT_005 | Confirm duplicate appointment time | Edge | Book same service and time twice | Duplicate appointment is prevented |"
    }
]

## Convert Dataset to JSONL Format
Since OpenAI fine-tuning requires JSONL data. Each line is a JSON object containing messages. I splitted the examples into a training file and a validation file.

In [29]:
# Each example is converted into a JSONL
def make_record(example):
    return {
        "messages": [
            {"role": "system", "content": system_message},
            {"role": "user", "content": example["requirement"]},
            {"role": "assistant", "content": example["answer"]}
        ]
    }

records = [make_record(ex) for ex in examples]

train_records = records[:12]
valid_records = records[12:]

train_path = PROJECT_DIR / "qa_testcase_train.jsonl"
valid_path = PROJECT_DIR / "qa_testcase_validation.jsonl"

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

write_jsonl(train_path, train_records)
write_jsonl(valid_path, valid_records)

#printing dataset information
print("Training file:", train_path)
print("Validation file:", valid_path)
print("Training examples:", len(train_records))
print("Validation examples:", len(valid_records))


Training file: AI-Powered-Test-Case-Generator/qa_testcase_train.jsonl
Validation file: AI-Powered-Test-Case-Generator/qa_testcase_validation.jsonl
Training examples: 12
Validation examples: 3


## Validation
I checked every line is valid JSON and contains system, user, and assistant messages.

In [30]:
# Validation
def validate_jsonl(path):
    errors = []
    count = 0

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            count += 1
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                errors.append(f"Line {line_number}: JSON error - {e}")
                continue

            if "messages" not in obj:
                errors.append(f"Line {line_number}: missing messages")
                continue

            roles = [m.get("role") for m in obj["messages"]]
            if roles != ["system", "user", "assistant"]:
                errors.append(f"Line {line_number}: roles are {roles}, expected ['system', 'user', 'assistant']")

    return count, errors

for path in [train_path, valid_path]:
    count, errors = validate_jsonl(path)
    print(f"{path.name}: {count} records")
    if errors:
        for err in errors:
            print("-", err)
    else:
        print("No validation errors.")


qa_testcase_train.jsonl: 12 records
No validation errors.
qa_testcase_validation.jsonl: 3 records
No validation errors.


## Preview Training Example
This preview confirmed, the model will receive a requirement and learn the desired structured response format.

In [31]:
# Preview One Training Example
with open(train_path, "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

print(json.dumps(first_line, indent=2))


{
  "messages": [
    {
      "role": "system",
      "content": "You are a QA test case generation assistant. Given a software requirement, generate concise, structured test cases. Use markdown table format with columns: TC ID, Scenario, Category, Steps, Expected Result."
    },
    {
      "role": "user",
      "content": "Login feature: username and password are required. Valid credentials allow login. Invalid credentials show an error. Account locks after five failed attempts."
    },
    {
      "role": "assistant",
      "content": "| TC ID | Scenario | Category | Steps | Expected Result |\n|---|---|---|---|---|\n| TC_LOGIN_001 | Login with valid credentials | Functional | Open login page; enter valid username; enter valid password; click Login | User logs in successfully and is redirected to dashboard |\n| TC_LOGIN_002 | Submit blank username and password | Negative | Open login page; leave username blank; leave password blank; click Login | Required field validation messages ar

## Upload Training and Validation Files
The files are uploaded with purpose="fine-tune". The returned file IDs were used to create the fine-tuning job.

In [32]:
## Upload Training and Validation Files
train_file = client.files.create(
    file=open(train_path, "rb"),
    purpose="fine-tune"
)

valid_file = client.files.create(
    file=open(valid_path, "rb"),
    purpose="fine-tune"
)

print("Training file ID:", train_file.id)
print("Validation file ID:", valid_file.id)


Training file ID: file-KQhRYTJRttbXAw82dnbMyA
Validation file ID: file-26sYmcrjdbx61cbXqadPcm


## Creating the Fine-Tuning Job
The fine-tuning job is created based on the uploaded data sets and the base model is trained by OpenAI on the provided QA examples to enable the model to understand the required style of the generated tests.

In [ ]:
# Create fine-tuning job
CREATE_FINE_TUNE_JOB = True 

if CREATE_FINE_TUNE_JOB:
    fine_tune_job = client.fine_tuning.jobs.create(
        model=BASE_MODEL,
        training_file=train_file.id,
        validation_file=valid_file.id,
        suffix="qa-testcase-generator"
    )
    job_id = fine_tune_job.id
    print("Fine-tuning job created.")
    print("Job ID:", job_id)
    print("Status:", fine_tune_job.status)
else:
    job_id = "ADD_JOB_ID_HERE"
    print("Fine-tuning job is not created in this run.")

Fine-tuning job created.
Job ID: ftjob-nciXUCZpM4ss7kUU9HiCtV77
Status: validating_files


## Retrieve Fine-Tuning Job Status
This tracks the job status, including whether it is validating files, queued, succeeded, failed etc.

In [48]:
# Retrieve fine-tuning job status
job = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", job.id)
print("Status:", job.status)
print("Base model:", job.model)
print("Fine-tuned model:", job.fine_tuned_model)
print("Training file:", job.training_file)
print("Validation file:", job.validation_file)
print("Trained tokens:", job.trained_tokens)
print("Created at:", job.created_at)
print("Finished at:", job.finished_at)

if job.error:
    print("Error:", job.error)


Job ID: ftjob-nciXUCZpM4ss7kUU9HiCtV77
Status: running
Base model: gpt-3.5-turbo-0125
Fine-tuned model: None
Training file: file-KQhRYTJRttbXAw82dnbMyA
Validation file: file-26sYmcrjdbx61cbXqadPcm
Trained tokens: None
Created at: 1778366257
Finished at: 1778367729
Error: Error(code=None, message=None, param=None)


## Tracking Job Events
This endpoint lists fine-tuning events. These events document what was happened during model build process.

In [49]:
# Tracking Job Events
events = client.fine_tuning.jobs.list_events(
    fine_tuning_job_id=job_id,
    limit=20
)

for event in events.data:
    print(event.created_at, "-", event.level, "-", event.message)

1778367734 - info - Evaluating model against our usage policies
1778367734 - info - New fine-tuned model created
1778367734 - info - Checkpoint created at step 84
1778367734 - info - Checkpoint created at step 72
1778367717 - info - Step 96/96: training loss=0.05, validation loss=0.81, full validation loss=0.70
1778367711 - info - Step 95/96: training loss=0.04, validation loss=0.52
1778367705 - info - Step 94/96: training loss=0.03, validation loss=0.76
1778367701 - info - Step 93/96: training loss=0.03, validation loss=0.51
1778367697 - info - Step 92/96: training loss=0.09, validation loss=0.77
1778367690 - info - Step 91/96: training loss=0.02, validation loss=0.81
1778367686 - info - Step 90/96: training loss=0.04, validation loss=0.76
1778367680 - info - Step 89/96: training loss=0.04, validation loss=0.52
1778367676 - info - Step 88/96: training loss=0.06, validation loss=0.80
1778367672 - info - Step 87/96: training loss=0.01, validation loss=0.77
1778367668 - info - Step 86/96

## Retrieve Metrics and Result Files
Once the fine-tuning job is succeded then the output files can be accessed to view critical data on training like job status, tokens trained, losses etc.

In [50]:
# retrieve job status until completion
job = client.fine_tuning.jobs.retrieve(job_id)

print("Status:", job.status)
print("Fine-tuned model:", job.fine_tuned_model)
print("Result files:", job.result_files)

if job.result_files:
    for result_file_id in job.result_files:
        print("\nDownloading result file:", result_file_id)
        try:
            result_content = client.files.content(result_file_id)

            if hasattr(result_content, "text"):
                text = result_content.text
            elif hasattr(result_content, "read"):
                text = result_content.read().decode("utf-8")
            else:
                text = str(result_content)

            print(text[:3000])
            result_path = PROJECT_DIR / f"{result_file_id}_results.txt"
            result_path.write_text(text, encoding="utf-8")
            print("Saved results to:", result_path)
        except Exception as e:
            print("Could not download or parse result file.")
            print("Error:", e)
else:
    print("No result files available yet. Wait until the job succeeds.")

Status: succeeded
Fine-tuned model: ft:gpt-3.5-turbo-0125:personal:qa-testcase-generator:DdkyUZ2Y
Result files: ['file-93VwHPjkrs66MgDrNMSTwi']

c3RlcCx0cmFpbl9sb3NzLHRyYWluX2FjY3VyYWN5LHZhbGlkX2xvc3MsdmFsaWRfbWVhbl90b2tlbl9hY2N1cmFjeQoxLDEuMTUwNjEsMC43NDIyNywxLjMyNTIsMC43NzUxNQoyLDEuNTQzNjEsMC43MzYyLDEuMjgzNzIsMC43ODc4OAozLDEuNTU4NjIsMC43NTkyNiwxLjQ2NjYyLDAuNzI1NjEKNCwxLjIwMzQyLDAuNzg1NzEsMS4xNzEyOSwwLjc5Mzk0CjUsMS40NTMxMywwLjc3Nzc4LDEuMTQyNjQsMC43ODEwNwo2LDEuNDIwNjQsMC43OTc0NywxLjI0NzcsMC43Mzc4CjcsMS4zNzgzOSwwLjcyNTYxLDEuMTQ1MywwLjczNzgKOCwwLjk1NTQsMC44MTAzNCwwLjg1MDQ1LDAuODEwNjUKOSwwLjgzODI4LDAuODE3NjUsMC43MDkyMSwwLjc3NTc2CjEwLDEuMTYxNjcsMC43NDg3MiwwLjYzNjg4LDAuNzg3ODgKMTEsMC43ODA3MiwwLjgwNTI2LDAuODY3ODYsMC43NjIyCjEyLDAuNTgzODUsMC43OTA3LDAuNjM1OTgsMC43OTg4MgoxMywwLjUyNjYsMC43OTY1MSwwLjUzNzk4LDAuODI0MjQKMTQsMC44MjY3MiwwLjc1NjEsMC44MDkwNCwwLjc4MDQ5CjE1LDAuNjEyOTcsMC43OTM4MSwwLjYwMTMsMC43OTg4MgoxNiwwLjU2OTM2LDAuODI3NTksMC41OTE3MSwwLjgwNDczCjE3LDAuNTkxNTYsMC44MzUyOSwwLjUwMDkyLDAuODQyNDI

## Fine-Tuning Evaluation
The fine-tuning job was successful with the use of the gpt-3.5-turbo-0125 model as the base. Metrics including training loss, validation loss, and token accuracy were obtained during the training process.Result shows, there is an improvement during training, meaning that the model learned the desired QA test case generation pattern.

## Testing the Fine-Tuned Model
After the fine-tuning process completed successfully, the model was tested using a new software requirement that was not part of the training dataset. The purpose of this test was to evaluate whether the model can develop structured and relevant QA test-cases that were unknown.

In [ ]:
# Testing the Fine-Tuned Model

job = client.fine_tuning.jobs.retrieve(job_id)

FINE_TUNED_MODEL = job.fine_tuned_model

test_requirement = """
Two-factor authentication feature:
After entering valid username & password, the user must enter a six-digit verification code.
Invalid code should show an error.
Expired code should be rejected.
After three invalid code attempts, verification should be blocked.
"""

response = client.chat.completions.create(
    model=FINE_TUNED_MODEL,
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": test_requirement}
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

| TC ID | Scenario | Category | Steps | Expected Result |
|---|---|---|---|---|
| TC_AUTH_001 | Verify valid two-factor authentication | Functional | Enter valid credentials; enter valid code | Logged in successfully |
| TC_AUTH_002 | Submit blank verification code | Negative | Enter valid credentials; leave code blank; submit | Required code message is displayed |
| TC_AUTH_003 | Enter invalid verification code | Negative | Enter valid credentials; enter invalid code; submit | Invalid code error is displayed |
| TC_AUTH_004 | Use expired verification code | Negative | Enter valid credentials; enter expired code; submit | Code expired message is displayed |
| TC_AUTH_005 | Reach maximum invalid attempts | Boundary | Enter valid credentials; enter wrong code three times | Verification is blocked after third attempt |
| TC_AUTH_006 | Enter code after block | Negative | Use blocked account credentials; enter valid code | Verification is rejected |


## Evaluation of Outupt
The tuned model managed to produce well-structured test cases for a two-factor authentication feature which was not part of the training dataset. Test cases followed the required QA format and covered various aspects such as functionality, negatives, and boundaries. This means that the model understood the required format for generating test cases through fine tuning.

## Conclusion
This milestone has been quite successful in demonstrating the method by which a tuned AI model is trained and tested for test case generation in QA tasks. The model was trained on a set of QA data and was capable of generating structured test cases for a new task that was not present in the training dataset.

The outputs have adhered to the expected format of QA test cases and included test scenarios from the perspectives of functionality, negativity, and boundaries, indicate the model has certainly learned the QA format through the fine-tuning process.

Also, this milestone has enabled hands-on experience in several important aspects, such as preparing datasets, formatting in JSONL format, uploading files to the server, launching fine-tuning jobs, tracking endpoints, getting metrics, and testing the model.